# 04: Cosinor Analysis

This notebook performs cosinor analysis to extract circadian rhythm parameters.

**CRITICAL METHODOLOGICAL CORRECTION:**

Cosinor is applied **AFTER behavioral state identification**, NOT to raw activity.

**Why raw activity cosinor is INCORRECT:**
- Raw activity mixes multiple behavioral states (rest, low activity, active)
- Cosinor assumes a single dominant rhythm, which is violated
- Results are biologically meaningless when states are mixed

**CORRECT APPROACH (implemented here):**
1. Apply cosinor to probability of being in active states: P(state ∈ {active states})
2. Compute state-conditioned mean activity cosinor
3. Calculate MESOR, amplitude, and acrophase per individual and per state

**Pipeline Order:** Raw activity → Features → HMM states → **COSINOR** → Environmental effects

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from loguru import logger

from src import io, cosinor

# Fix matplotlib overflow error with large datasets
plt.rcParams['agg.path.chunksize'] = 10000

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')

## 1. Load Data

In [ ]:
df = io.load_raw_data(Path('../data/processed/hmm_data.parquet'))
logger.info(f"Loaded data with shape: {df.shape}")

# Verify HMM states are present
if 'hmm_state' not in df.columns:
    logger.error("No HMM states found! Run 03_hmm_hsmm.ipynb first.")
    raise ValueError("Missing HMM states - cosinor requires behavioral states")

logger.success("HMM states found - ready for state-based cosinor")
print(f"\nUnique states: {sorted(df['hmm_state'].unique())}")
if 'state_label' in df.columns:
    print(f"State labels: {df[['hmm_state', 'state_label']].drop_duplicates().to_dict('records')}")

df.head()

## 2. State-Based Cosinor Analysis

This is the CORRECT approach: analyze rhythms in behavioral states, not raw mixed activity.

In [ ]:
# Fit cosinor per behavioral state
# This computes:
# 1. Cosinor on P(active state) - probability of being in active states
# 2. Per-state cosinor on mean activity

state_cosinor = cosinor.fit_cosinor_per_state(
    df,
    time_column='timestamp',
    state_column='hmm_state',
    activity_column='activity',
    period=24.0,
    active_states=None  # Auto-detect from mean activity
)

print("\n" + "="*80)
print("ACTIVE STATE PROBABILITY COSINOR")
print("="*80)
print(f"Active states identified: {state_cosinor['active_states']}")
print(f"MESOR (mean P(active)): {state_cosinor['active_probability_cosinor']['mesor']:.3f}")
print(f"Amplitude: {state_cosinor['active_probability_cosinor']['amplitude']:.3f}")
print(f"Acrophase: {state_cosinor['active_probability_cosinor']['acrophase_hours']:.2f} hours")
print(f"R²: {state_cosinor['active_probability_cosinor']['r_squared']:.3f}")

print("\n" + "="*80)
print("PER-STATE COSINOR PARAMETERS")
print("="*80)
for state, params in state_cosinor['per_state_cosinor'].items():
    label = df[df['hmm_state']==state]['state_label'].iloc[0] if 'state_label' in df.columns else f"State {state}"
    print(f"\n{label} (State {state}):")
    print(f"  Mean Activity: {state_cosinor['state_mean_activity'][state]:.1f}")
    print(f"  MESOR: {params['mesor']:.3f}")
    print(f"  Amplitude: {params['amplitude']:.3f}")
    print(f"  Acrophase: {params['acrophase_hours']:.2f} hours")
    print(f"  R²: {params['r_squared']:.3f}")

## 3. Per-Individual Per-State Cosinor

Compute cosinor parameters for each individual and each state for population-level analysis.

In [ ]:
# Fit cosinor per individual per state
if 'subject' in df.columns:
    individual_cosinor = cosinor.fit_cosinor_per_individual_per_state(
        df,
        time_column='timestamp',
        subject_column='subject',
        state_column='hmm_state',
        activity_column='activity',
        period=24.0
    )
    
    print("\nPER-INDIVIDUAL PER-STATE COSINOR RESULTS:")
    print(individual_cosinor.head(20))
    
    # Summary statistics
    print("\nSUMMARY BY STATE:")
    summary = individual_cosinor.groupby('state')[['mesor', 'amplitude', 'acrophase_hours', 'r_squared']].agg(['mean', 'std'])
    print(summary)
    
    logger.success("Individual-level cosinor computed")
else:
    logger.warning("No subject column - skipping per-individual analysis")
    individual_cosinor = None

## 4. Visualize State-Based Rhythms

Visualize circadian patterns in behavioral states.

In [ ]:
# Visualize hourly patterns by state
import matplotlib.pyplot as plt
import seaborn as sns

df_plot = df.copy()
df_plot['hour'] = pd.to_datetime(df_plot['timestamp']).dt.hour

# Plot mean activity by hour and state
hourly_by_state = df_plot.groupby(['hour', 'hmm_state'])['activity'].mean().reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Activity by state
for state in sorted(df_plot['hmm_state'].unique()):
    state_data = hourly_by_state[hourly_by_state['hmm_state'] == state]
    label = df_plot[df_plot['hmm_state']==state]['state_label'].iloc[0] if 'state_label' in df_plot.columns else f"State {state}"
    axes[0].plot(state_data['hour'], state_data['activity'], marker='o', label=label)

axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Mean Activity')
axes[0].set_title('Mean Activity by Hour and State')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(range(0, 24, 2))

# Plot 2: State prevalence by hour
state_prev = df_plot.groupby('hour')['hmm_state'].value_counts(normalize=True).reset_index()
state_prev.columns = ['hour', 'state', 'prevalence']

for state in sorted(df_plot['hmm_state'].unique()):
    state_data = state_prev[state_prev['state'] == state]
    label = df_plot[df_plot['hmm_state']==state]['state_label'].iloc[0] if 'state_label' in df_plot.columns else f"State {state}"
    axes[1].plot(state_data['hour'], state_data['prevalence'], marker='o', label=label)

axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('State Prevalence')
axes[1].set_title('Behavioral State Prevalence by Hour (Diel Pattern)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.savefig('../outputs/state_based_rhythms.png', dpi=300, bbox_inches='tight')
plt.show()

logger.success("State-based rhythm visualizations created")

## 4. Multi-Period Analysis

In [ ]:
# Test multiple periods
df_clean = df.dropna(subset=['activity', 'timestamp'])
t_start = pd.to_datetime(df_clean['timestamp']).min()
t_hours = (pd.to_datetime(df_clean['timestamp']) - t_start).dt.total_seconds() / 3600

multi_results = cosinor.fit_multi_period_cosinor(
    t_hours.values,
    df_clean['activity'].values,
    periods=[24.0, 12.0]
)

print("\nMulti-Period Cosinor Results:")
for period, results in multi_results.items():
    print(f"\nPeriod {period}h:")
    print(f"  MESOR: {results['mesor']:.3f}")
    print(f"  Amplitude: {results['amplitude']:.3f}")
    print(f"  R²: {results['r_squared']:.3f}")

## 5. Test Rhythmicity Significance

In [ ]:
# Test statistical significance (using fewer permutations for speed)
rhythmicity_test = cosinor.test_rhythmicity(
    t_hours.values,
    df_clean['activity'].values,
    period=24.0,
    n_permutations=100,  # Increase for more robust results
    random_state=RANDOM_STATE
)

print("\nRhythmicity Test:")
print(f"  Actual Amplitude: {rhythmicity_test['actual_amplitude']:.3f}")
print(f"  P-value: {rhythmicity_test['p_value']:.4f}")
print(f"  Significant (p<0.05): {rhythmicity_test['significant']}")

## 6. Save Results

In [ ]:
# Add state-based cosinor results to dataframe
# Note: These are state-level parameters, not individual time points

# Save per-individual per-state cosinor if available
if individual_cosinor is not None:
    output_path = Path('../data/processed/cosinor_per_individual_state.parquet')
    io.save_processed_data(individual_cosinor, output_path, format='parquet')
    logger.success(f"Per-individual-state cosinor saved to {output_path}")

# Save the main dataframe with states
output_path = Path('../data/processed/cosinor_data.parquet')
io.save_processed_data(df, output_path, format='parquet')
logger.success(f"Data with HMM states saved to {output_path}")

In [ ]:
params = {
    'random_state': RANDOM_STATE,
    'period': 24.0,
    'method': 'state_based_cosinor',
    'active_states': state_cosinor['active_states'],
    'n_states': len(state_cosinor['per_state_cosinor']),
    'per_individual': individual_cosinor is not None,
}
logger.info(f"Cosinor parameters: {params}")
logger.success("State-based cosinor analysis complete - biologically valid approach used")